# Subbasin masks on the Eocene PALEORCA2 domain

`ecefast.components.nemo.domain.subbasins.generate_subbasin_masks` classifies every ocean point of a NEMO
`mesh_mask_<name>.nc` into Atlantic, Pacific or Indian with fixed lon/lat polygons drawn on **present-day**
geography (task 5.3, issue #16). This notebook runs it on the Eocene domain and plots every mask on the Eocene
land-sea mask, to judge by eye whether the basins make sense for Eocene geography. The present-day domain is shown
for reference.

**Decision (task 5.3, PR #39):** they do not (see the assessment at the end). Until Eocene basins are defined in a
separate agreed task, the domain pipeline writes the **global mask only** (`glomsk`) for paleo bathymetries:
`EoceneBathymetry.regional_subbasins = False`. This notebook runs the regional masks anyway
(`regional=True`, the default) to keep the record that the future task starts from.

DOMAINcfg output (`mesh_mask_<name>.nc`) exists only on Atos, so a minimal `mesh_mask` (`tmask`, `glamt`, `gphit`)
is rebuilt from the post-processed domain (`domain_cfg.nc`): the ocean levels of a column are `1..bottom_level`.
For present day this reproduces the Atos-made `subbasins.nc` exactly.

**Inputs:** `ECEFAST_TEST_DATA` (as for `pytest -m local_data`): a directory holding
`data/domain/PALEORCA2/{present_day,eocene}/domain_cfg.nc`. Optional `ECEFAST_SUBBASINS_FIG`: a directory where
the figures are saved as PNG.

In [ ]:
import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.colors import ListedColormap
from scipy import ndimage

from ecefast.components.nemo.domain.subbasins import generate_subbasin_masks

DATA = Path(os.environ["ECEFAST_TEST_DATA"])  # or set the path here; do not commit personal paths
DOMAINS = DATA / "data" / "domain" / "PALEORCA2"
FIGDIR = os.environ.get("ECEFAST_SUBBASINS_FIG")
WORK = Path(tempfile.mkdtemp(prefix="subbasins-"))

In [ ]:
def rebuild_mesh_mask(name: str) -> None:
    """Write WORK/mesh_mask_<name>.nc with the variables that generate_subbasin_masks reads."""
    with xr.open_dataset(DOMAINS / name / "domain_cfg.nc") as domain:
        levels = np.arange(1, domain.sizes["z"] + 1)
        tmask = (levels[None, :, None, None] <= domain["bottom_level"].values[:, None]).astype("int8")
        mesh = xr.Dataset(
            {
                "tmask": (("time_counter", "nav_lev", "y", "x"), tmask),
                "glamt": (("time_counter", "y", "x"), domain["glamt"].values),
                "gphit": (("time_counter", "y", "x"), domain["gphit"].values),
            }
        )
        mesh.to_netcdf(WORK / f"mesh_mask_{name}.nc")


masks = {}
for name in ("present_day", "eocene"):
    rebuild_mesh_mask(name)
    generate_subbasin_masks(str(WORK), str(WORK / name), name)
    masks[name] = xr.load_dataset(WORK / name / "subbasins.nc")

## Checks: present-day identity, coverage, overlaps

Every ocean point should be in exactly one of Atlantic, Pacific, Indian; the method leaves points outside all
polygons unassigned. The table lists the connected groups of unassigned ocean points (3 points or more).

In [ ]:
atos = DOMAINS / "present_day" / "subbasins.nc"
if atos.is_file():
    xr.testing.assert_identical(xr.load_dataset(atos), masks["present_day"])
    print("present_day: identical to the Atos-made subbasins.nc")

for name, ds in masks.items():
    ocean = ds["glomsk"].values == 1
    basins = ds["atlmsk"].values + ds["pacmsk"].values + ds["indmsk"].values
    print(f"\n{name}: {ocean.sum()} ocean points")
    for var in ("atlmsk", "pacmsk", "indmsk"):
        print(f"  {var}: {int(ds[var].sum())}")
    print(f"  in two basins: {int((basins > 1).sum())}, on land: {int((basins[~ocean] > 0).sum())}")
    unassigned = ocean & (basins == 0)
    labels, count = ndimage.label(unassigned)
    print(f"  unassigned: {int(unassigned.sum())}")
    groups = [labels == i for i in range(1, count + 1)]
    for group in sorted(groups, key=lambda g: -g.sum()):
        if group.sum() >= 3:
            lon, lat = ds["navlon"].values[group], ds["navlat"].values[group]
            print(
                f"    {group.sum():4d} points, lon {lon.min():7.1f} .. {lon.max():6.1f}, "
                f"lat {lat.min():6.1f} .. {lat.max():5.1f}"
            )

## Maps

One panel per mask (coloured points), on the land-sea mask of the domain (grey = land); the last panel shows the
classification, with unassigned ocean points in red. Points are plotted at their lon/lat (`glamt`, `gphit`):
north of about 20°N the PALEORCA2 grid is not regular, so the points thin out or bunch up there.

In [ ]:
PANELS = [
    ("atlmsk", "Atlantic"),
    ("pacmsk", "Pacific"),
    ("indmsk", "Indian"),
    ("indpacmsk", "Indo-Pacific"),
    ("glomsk", "Global"),
]
CLASSES = ListedColormap(["#d9d9d9", "#9ecae1", "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"])


def plot_masks(name: str) -> plt.Figure:
    ds = masks[name]
    lon, lat = ds["navlon"].values.ravel(), ds["navlat"].values.ravel()
    ocean = ds["glomsk"].values.ravel() == 1
    fig, axes = plt.subplots(3, 2, figsize=(14, 13), sharex=True, sharey=True, layout="constrained")
    for ax, (var, title) in zip(axes.flat, PANELS):
        inside = ds[var].values.ravel() == 1
        ax.scatter(lon[~ocean], lat[~ocean], s=3, marker="s", c="#bdbdbd", linewidths=0)
        ax.scatter(lon[ocean & ~inside], lat[ocean & ~inside], s=3, marker="s", c="#deebf7", linewidths=0)
        ax.scatter(lon[inside], lat[inside], s=3, marker="s", c="#1f77b4", linewidths=0)
        ax.set_title(f"{title} ({var}): {inside.sum()} points")
    # classification: 0 land, 1 unassigned ocean (red), 2 Atlantic, 3 Pacific, 4 Indian
    classes = np.zeros(lon.shape, dtype=int)
    classes[ocean] = 5
    for value, var in ((2, "atlmsk"), (3, "pacmsk"), (4, "indmsk")):
        classes[ds[var].values.ravel() == 1] = value
    ax = axes.flat[-1]
    ax.scatter(lon, lat, s=3, marker="s", c=classes, cmap=CLASSES, vmin=0, vmax=5, linewidths=0)
    ax.set_title("Atlantic (blue), Pacific (orange), Indian (green), unassigned (red)")
    for ax in axes.flat:
        ax.set_xlim(-180, 180)
        ax.set_ylim(-90, 90)
        ax.set_xticks(range(-180, 181, 60))
        ax.set_yticks(range(-90, 91, 30))
        ax.grid(lw=0.3)
    fig.suptitle(f"Subbasin masks, PALEORCA2 {name}")
    if FIGDIR:
        fig.savefig(Path(FIGDIR) / f"subbasins_{name}.png", dpi=110)
    return fig


plot_masks("eocene");

In [ ]:
plot_masks("present_day");

## Assessment (Eocene domain built on Atos, task 5.3)

What the present-day polygons give on the Eocene domain (lon/lat are approximate, from the maps and the checks
above):

- **Coverage.** No ocean point is in two basins and no land point is in a basin; `glomsk` is the ocean mask.
  806 Eocene ocean points (4.8 %) are unassigned, against 609 (3.6 %) for present day.
- **North-east Pacific off paleo North America** (about 147–102°W, 21–68°N, 256 points): open Pacific water is
  unassigned. The east edge of the Pacific polygon and the west edge of the Atlantic polygon follow today's North
  American coast; the paleo coast lies further west.
- **Eastern Pacific off paleo South America** (about 72–58°W, 5°N–60°S): Pacific water west of South America is
  in the **Atlantic**. The Atlantic–Pacific line (74°W, 7.5°N → 66°W, 81°S) follows today's Andes, while the
  paleo continent lies 6–14° further east.
- **Central American Seaway** (about 102–84°W, 8–21°N): open in the Eocene; it is cut along today's Central
  American coast. A cut there is needed anyway, but its position is arbitrary.
- **Tethys.** West of a line 20°E, 14°N → 44°E, 35°N the Tethys is Atlantic, east of it Indian: the cut
  (about 20–42°E, 14–32°N) runs through open water along today's Red Sea. The Paratethys / Turgai region (about
  48–75°E, 36–57°N, 79 points) is unassigned, while the West Siberian Sea further north is Atlantic.
- **Indian–Pacific boundary** (about 98–142°E, 37°S–5°N): with no Indonesian archipelago, the line
  100°E, 1°N → 142°E, 27°S cuts through open water north of Australia; a strip east of paleo-Australia (about
  138–142°E, 35°S) is Indian.
- **Arctic and Bering sector** (north of about 58°N, 93°E–100°W, 410 points): unassigned, as for present day
  (the Atlantic sector of the Arctic is Atlantic). In the Eocene the Arctic is nearly isolated, so including part
  of it in the Atlantic is questionable.
- **Southern Ocean**: the meridional cuts (about 68°W, 25°E, 146°E) are conventional and look as reasonable as
  for present day.
- **Atlantic proper** (closed to the north apart from the Arctic gateways, open to the Southern Ocean) and the
  **Indian Ocean proper** are recognisable.

Options considered:

- (a) Eocene-specific polygons (in `paleo/eocene/`), with the generic code taking the polygons as a parameter.
- (b) Basins built from the paleo coastline: flood fill from seed points per basin, with explicit cuts only at
  the gateways (Central American Seaway, Tethys, Southern Ocean meridians).
- (c) Only `glomsk` for paleo bathymetries.

**Decision:** (c) for now; (a) or (b) is a separate task, to be agreed by both authors. The present-day polygons
and method are unchanged.